# OpenWebUI + Ollama on Kaggle

这是一份全新 notebook（不修改你原有文件），用于在 Kaggle 里运行 OpenWebUI + Ollama。

按顺序执行下面代码单元：
1. 安装依赖
2. 启动 Ollama
3. 拉取模型并测试
4. 启动 OpenWebUI
5. 用 ngrok 暴露 OpenWebUI 公网访问地址

注意：Kaggle Session 重启后需要重新运行。

In [ ]:
# 1) 安装依赖
import subprocess


def run(cmd: str):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, check=True)

run("apt-get update -y")
run("apt-get install -y curl wget git zstd")
run("curl -fsSL https://ollama.com/install.sh | sh")
run("python -m pip -q install --upgrade pip")
run("python -m pip -q install open-webui")

print("\n依赖安装完成")

In [ ]:
# 2) 启动 Ollama 服务
import os
import subprocess
import time
from pathlib import Path

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
log_dir = Path("/kaggle/working/logs")
log_dir.mkdir(parents=True, exist_ok=True)

ollama_log = open(log_dir / "ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# 等待服务就绪
time.sleep(5)
print(f"Ollama PID: {ollama_proc.pid}")
print("Ollama endpoint: http://127.0.0.1:11434")
print("日志文件: /kaggle/working/logs/ollama.log")

In [ ]:
# 3) 拉取模型并做最小测试
import subprocess

MODEL_NAME = "gemma4:e4b"  # 可改成其他模型，例如 llama3.2:1b

subprocess.run(f"ollama pull {MODEL_NAME}", shell=True, check=True)
print("模型拉取完成:", MODEL_NAME)

subprocess.run(
    f"ollama run {MODEL_NAME} \"Reply with: ollama is ready\"",
    shell=True,
    check=True,
)

In [ ]:
# 4) 启动 OpenWebUI
import os
import sys
import time
import socket
import shutil
import subprocess
import urllib.request
from pathlib import Path

os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
os.environ["WEBUI_AUTH"] = "False"  # 演示时关闭登录
os.environ["DATA_DIR"] = "/kaggle/working/openwebui-data"  # 避免写入受限目录
os.environ["USER_AGENT"] = "kaggle-openwebui-setup"

log_dir = Path("/kaggle/working/logs")
log_dir.mkdir(parents=True, exist_ok=True)
Path(os.environ["DATA_DIR"]).mkdir(parents=True, exist_ok=True)

# 选一个可用端口（优先 8080）
def pick_port(candidates):
    for p in candidates:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.5)
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    return 8080

openwebui_port = pick_port([8080, 8081, 8082, 18080])
os.environ["OPENWEBUI_PORT"] = str(openwebui_port)

openwebui_bin = shutil.which("open-webui")
if openwebui_bin is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open-webui"], check=True)
    openwebui_bin = shutil.which("open-webui")
    if openwebui_bin is None:
        raise RuntimeError("open-webui executable not found after installation")

openwebui_log_path = log_dir / "openwebui.log"
webui_log = open(openwebui_log_path, "w")
webui_proc = subprocess.Popen(
    [openwebui_bin, "serve", "--host", "0.0.0.0", "--port", str(openwebui_port)],
    stdout=webui_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# 仅做简短探活，不阻断后续步骤
ready = False
for _ in range(20):
    if webui_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{openwebui_port}", timeout=2) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

print(f"OpenWebUI PID: {webui_proc.pid}")
print(f"本地地址: http://127.0.0.1:{openwebui_port}")
print("日志文件: /kaggle/working/logs/openwebui.log")
print("OpenWebUI quick-ready:", ready)

if webui_proc.poll() is not None:
    print("OpenWebUI 进程已退出，退出码:", webui_proc.returncode)
    try:
        tail = openwebui_log_path.read_text(errors="ignore").splitlines()[-120:]
        print("\n===== openwebui.log tail =====")
        print("\n".join(tail))
    except Exception:
        pass
else:
    print("OpenWebUI 正在后台继续初始化（首次运行可能需要数分钟）。")

In [ ]:
# 5) 启动 ngrok 暴露 OpenWebUI 公网访问地址
import os
import sys
import time
import socket
import subprocess
import urllib.request
import urllib.error

Ngrok_token = "21JxiosD62PLsE9BJL6AQqZRqkF_7fw6tKfScfb8aureupPzE"
OPENWEBUI_PORT = int(os.environ.get("OPENWEBUI_PORT", "8080"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok==6.1.0"], check=True)
from pyngrok import ngrok

# 等待 OpenWebUI 本地端口与 HTTP 就绪
local_ready = False
for _ in range(900):
    tcp_up = False
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        if s.connect_ex(("127.0.0.1", OPENWEBUI_PORT)) == 0:
            tcp_up = True

    if tcp_up:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{OPENWEBUI_PORT}", timeout=3) as resp:
                if resp.status < 500:
                    local_ready = True
                    break
        except urllib.error.HTTPError as e:
            # 4xx 说明服务已响应（例如未登录或权限控制）
            if e.code < 500:
                local_ready = True
                break
        except Exception:
            pass

    time.sleep(1)

if not local_ready:
    raise RuntimeError(
        f"OpenWebUI local service not ready on port {OPENWEBUI_PORT}. Run cell 6 and inspect openwebui.log first."
    )

# 建立 ngrok 隧道
ngrok.set_auth_token(Ngrok_token)
ngrok.kill()
public_tunnel = ngrok.connect(OPENWEBUI_PORT)
public_url = public_tunnel.public_url

# 再检查公网 URL 是否可达，避免拿到立刻报错的链接
public_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen(public_url, timeout=6) as resp:
            if resp.status < 500:
                public_ready = True
                break
    except urllib.error.HTTPError as e:
        if e.code < 500:
            public_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

print("Ngrok OpenWebUI URL:")
print(public_url)
print(f"Mapped local port: {OPENWEBUI_PORT}")
print("OpenWebUI local ready:", local_ready)
print("Ngrok public ready:", public_ready)

if not public_ready:
    print("注意：隧道已建立，但公网入口还在预热。等 30-90 秒后刷新 URL。")
    print("如果浏览器出现 ngrok 安全提示页，点击 Continue 即可进入 OpenWebUI。")

In [ ]:
# 6) 故障排查：查看日志与端口
!echo "===== ollama.log ====="
!tail -n 120 /kaggle/working/logs/ollama.log
!echo "===== openwebui.log ====="
!tail -n 200 /kaggle/working/logs/openwebui.log
!echo "===== listening ports (11434/8080/8081/8082/18080) ====="
!ss -lntp | grep -E "11434|8080|8081|8082|18080" || true
!echo "===== open-webui process ====="
!ps -ef | grep -i "open-webui" | grep -v grep || true

## 7) 保活（可选）

如果你需要长时间通过 ngrok 访问 OpenWebUI，请运行下面单元保持会话活跃。

说明：
- 在 Kaggle 中，如果会话结束（比如版本运行完成后变成 Viewer 状态），后台进程和 ngrok 隧道会被回收。
- 该单元会定时输出心跳，按 `Interrupt` 可随时停止。

In [ ]:
# 7) 保活（可选）
import os
import time
import urllib.request

OPENWEBUI_PORT = int(os.environ.get("OPENWEBUI_PORT", "8080"))
print(f"Keepalive started. Checking http://127.0.0.1:{OPENWEBUI_PORT} every 30s")
print("Press Interrupt to stop this cell.")

while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{OPENWEBUI_PORT}", timeout=5) as resp:
            print(time.strftime("%H:%M:%S"), "alive", "status=", resp.status)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "check_failed", str(e))
    time.sleep(30)

## 8) 暴露 Ollama API（公网）

此单元会新建一条 ngrok 隧道，把 Ollama 的 `11434` 端口直接暴露给外网。

说明：
- 这是给外部程序直接调用 Ollama API 用的（不是 OpenWebUI 页面）。
- 运行后会打印 `Ollama API URL` 与示例调用命令。
- 不用时请关闭 Kaggle 实例，避免公网可访问。

In [ ]:
# 8) 暴露 Ollama API（公网）
import os
import sys
import time
import socket
import subprocess
import urllib.request
import urllib.error

# 复用同一 ngrok token
Ngrok_token = "21JxiosD62PLsE9BJL6AQqZRqkF_7fw6tKfScfb8aureupPzE"
OLLAMA_PORT = 11434

# 安装并导入 pyngrok
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok==6.1.0"], check=True)
from pyngrok import ngrok

# 检查本地 Ollama 是否可用
local_ready = False
for _ in range(30):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1)
            if s.connect_ex(("127.0.0.1", OLLAMA_PORT)) == 0:
                local_ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if not local_ready:
    raise RuntimeError("Ollama is not listening on 11434. Please rerun cell 2 first.")

# 不 kill 全部隧道，避免影响 OpenWebUI 的 ngrok URL
ngrok.set_auth_token(Ngrok_token)
ollama_tunnel = ngrok.connect(OLLAMA_PORT)
ollama_api_url = ollama_tunnel.public_url

# 简单探测 API
remote_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen(f"{ollama_api_url}/api/tags", timeout=6) as resp:
            if resp.status < 500:
                remote_ready = True
                break
    except urllib.error.HTTPError as e:
        if e.code < 500:
            remote_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

print("Ollama API URL:")
print(ollama_api_url)
print("Remote ready:", remote_ready)
print("\nTest endpoints:")
print(f"- {ollama_api_url}/api/tags")
print(f"- {ollama_api_url}/api/generate")

print("\nExample curl (Linux/Mac):")
print(
    "curl -s "
    + f"{ollama_api_url}/api/generate "
    + "-d '{\"model\":\"gemma4:e4b\",\"prompt\":\"hello\",\"stream\":false}'"
)

print("\nExample Python:")
print(
    "import requests\n"
    + f"url = '{ollama_api_url}/api/generate'\n"
    + "payload = {'model': 'gemma4:e4b', 'prompt': 'hello', 'stream': False}\n"
    + "print(requests.post(url, json=payload, timeout=60).json())"
)

if not remote_ready:
    print("\n注意：隧道已创建，但 API 可能还在预热，等 10-30 秒后再请求。")